In [1]:
from calibpipe.tools.muon_throughput_calculator import CalculateThroughputWithMuons
import yaml
from traitlets.config import Config
from calibpipe.database.connections import CalibPipeDatabase
from calibpipe.database.interfaces import TableHandler
from calibpipe.throughput.throughput_containers import (
    OpticalThoughtputContainer,
)
import yaml
import pytest
import numpy as np
import sqlalchemy as sa

from datetime import datetime
from pathlib import Path
from traitlets.config.loader import Config

from calibpipe.throughput.throughput_containers import (
    OpticalThoughtputContainer,
)
from calibpipe.database.adapter.database_containers import (
    optical_throughput_sql_info
)
from calibpipe.tools.muon_throughput_calculator import CalculateThroughputWithMuons
from calibpipe.database.connections import CalibPipeDatabase
from calibpipe.database.interfaces import TableHandler
yaml_file_path = '/Users/vdk/Software/ctasoft/calibpipe/doc/source/examples/throughput/configurations/processor_tool_muon_configuration.yaml'
yaml_file_path = '/Users/vdk/software/ctasoft/calibpipe/doc/source/examples/throughput/configurations/throughput_muon_configuration_localdb.yaml'

def dict_to_config(d):
    """Recursively convert a dictionary into a Config object."""
    config = Config()
    for key, value in d.items():
        # If the key starts with an uppercase letter, ensure the value is a Config instance
        if key[0].isupper() and isinstance(value, dict):
            config[key] = dict_to_config(value)
        else:
            config[key] = value
    return config

def load_config_from_yaml(yaml_file_path):
    # Load the YAML file
    with open(yaml_file_path, 'r') as file:
        yaml_data = yaml.safe_load(file)
    
    # Convert the YAML data (a dictionary) to a Config object
    config = dict_to_config(yaml_data)
    return config

# Example usage:

config = load_config_from_yaml(yaml_file_path)

#config['DataWriter']['output_path'] = '/Users/vdk/Software/ctasoft/calibpipe/src/calibpipe/tests/data/throughput/notempy.h5'
#config['DataWriter']['output_key'] = 'muons'

#config['EventSource']['input_url'] = '/Users/vdk/Software/ctasoft/calibpipe/src/calibpipe/tests/data/throughput/Dummy100_NSB.simtel'
#config['EventSource']['input_url'] =  filename


config['CalculateThroughputWithMuons']['input_file'] = '/Users/vdk/software/ctasoft/calibpipe/src/calibpipe/tests/data/throughput/lst_muon_table.h5'
config['CalculateThroughputWithMuons']['output_folder'] = '/Users/vdk/software/ctasoft/calibpipe/src/calibpipe/tests/data/throughput/output'

In [2]:
config

{'database_configuration': {'user': 'test_user',
  'password': 'test_password',
  'database': 'test_calibpipe_db',
  'host': 'localhost',
  'port': 5432,
  'autocommit': True},
 'CalculateThroughputWithMuons': {'input_file': '/Users/vdk/software/ctasoft/calibpipe/src/calibpipe/tests/data/throughput/lst_muon_table.h5',
  'output_file': 'None',
  'output_folder': '/Users/vdk/software/ctasoft/calibpipe/src/calibpipe/tests/data/throughput/output',
  'statistic': 100,
  'min_ring_radius': 0.8,
  'max_ring_radius': 1.5,
  'min_impact_parameter': 0.2,
  'max_impact_parameter': 0.9,
  'ring_completeness_threshold': 0.3,
  'intensity_ratio': 0.5,
  'ring_containment_threshold': 0.5},
 'ReferenceMetadataContainer': {'version': '0.1'},
 'ProductReferenceMetadataContainer': {'description': 'Absolute optical efficiency coefficient',
  'creation_time': '01.01.2024 00:00:00',
  'product_id': '123456789',
  'data_category': 'B',
  'data_level': 'DL1',
  'data_association': 'Telescope',
  'data_type': 

In [3]:
muon_processor_tool = CalculateThroughputWithMuons(config=config)

muon_processor_tool.setup()

In [4]:
muon_processor_tool.start()

In [5]:
muon_processor_tool.container_dict['tel_001'].statistic

3

In [6]:
muon_processor_tool.finish()

2025-01-16 15:14:58,518 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-01-16 15:14:58,518 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-01-16 15:14:58,519 INFO sqlalchemy.engine.Engine select current_schema()
2025-01-16 15:14:58,520 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-01-16 15:14:58,520 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-01-16 15:14:58,520 INFO sqlalchemy.engine.Engine [raw sql] {}


2025-01-16 15:14:58,528 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-01-16 15:14:58,529 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-01-16 15:14:58,530 INFO sqlalchemy.engine.Engine select current_schema()
2025-01-16 15:14:58,530 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-01-16 15:14:58,530 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-01-16 15:14:58,531 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-01-16 15:14:58,533 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-01-16 15:14:58,534 INFO sqlalchemy.engine.Engine INSERT INTO optical_throughput (tel_id, obs_id, validity_start, validity_end, optical_throughput_coefficient, optical_throughput_coefficient_std, method, statistic) VALUES (%(tel_id)s::INTEGER, %(obs_id)s::INTEGER, %(validity_start)s::TIMESTAMP WITH TIME ZONE, %(validity_end)s::TIMESTAMP WITH TIME ZONE, %(optical_throughput_coefficient)s, %(optical_throughput_coefficient_std)s, %(method)s::VARCHAR, %(statistic)s::INTEGER) R

In [ ]:
db_config_path = "/Users/vdk/Software/ctasoft/calibpipe/doc/source/examples/utils/configuration/db_config.yaml"
with open(db_config_path) as yaml_file:    
    db_data = yaml.safe_load(yaml_file)
with CalibPipeDatabase(
    **db_data["database_configuration"],
) as connection:
    qtable = TableHandler.read_table_from_database(
        type(OpticalThoughtputContainer()), connection
    )
    print("QTABLE = ", qtable)

In [ ]:
from astropy.time import Time
Time(qtable['validity_start'])

In [ ]:
qtable['validity_end'] = Time(qtable['validity_end'])
qtable['validity_end']

In [41]:
rows_as_dicts = [dict(row) for row in qtable[:-2]]

In [ ]:
rows_as_dicts[:]

In [ ]:
qtable[-1]['tel_id']

In [ ]:
with CalibPipeDatabase(
    **db_data["database_configuration"],
) as connection:
    db_table = optical_throughput_sql_info
    # Put the query in descending order to always fetch the last one record
    query = sa.select(db_table.get_table()).order_by(
        sa.desc(db_table.get_table().c.ID)
    )
    query_result = connection.execute(query).first()
    # Extract from DB field all columns except of first column that is autoincremented  value 
    # of primary key `ID`, because it is not related to the functionality that is tested here.
    query_result_dict = query_result._asdict()

In [ ]:
query_result.tel_id